# P6 Empirical Network Behavioral Evidence Model: Colab Training Workflow
### SIH 2026 Problem Statement 26105 | Team Nano X

This standalone, self-contained notebook runs the **exact same preprocessing, leakage elimination, IV analysis, feature schema, model architecture, and evaluation** as the local pipeline on Google Colab with an **NVIDIA T4 GPU**.

**Key Highlights:**
- Uses **NVIDIA T4 GPU** (`device='cuda'`, `tree_method='hist'`) for high-throughput gradient boosting.
- Removes testbed leakage (`Destination Port`), duplicate columns (`Fwd Header Length.1`), and 8 true constant columns.
- Normalizes all 15 attack classes from the official CIC-IDS2017 dataset into binary target while preserving multi-class labels for sub-category auditing.
- Generates identical metrics and produces `CyberOptRQ_P6_CIC2017_XGBoost_v1.pkl` and `metadata.json`.

## 1. Environment & GPU Verification

In [ ]:
!nvidia-smi
import psutil, os, sys, json, time, glob
import numpy as np
import pandas as pd
import xgboost as xgb
import sklearn

print("=" * 60)
print("COMPUTE & ENVIRONMENT DETECTION")
print("=" * 60)
print(f"Python Version: {sys.version}")
print(f"XGBoost Version: {xgb.__version__}")
print(f"Scikit-Learn Version: {sklearn.__version__}")
vm = psutil.virtual_memory()
print(f"System RAM: {vm.total / (1024**3):.2f} GB Total, {vm.available / (1024**3):.2f} GB Available")

# Verify GPU device compatibility
try:
    test_x = np.random.randn(100, 10).astype(np.float32)
    test_y = np.random.randint(0, 2, size=100).astype(np.int32)
    test_clf = xgb.XGBClassifier(n_estimators=2, max_depth=2, tree_method='hist', device='cuda')
    test_clf.fit(test_x, test_y)
    print("\n[SUCCESS] XGBoost GPU acceleration (device='cuda') verified on Colab GPU!")
    DEVICE = 'cuda'
except Exception as e:
    print(f"\n[WARNING] GPU test error ({e}). Fallback to CPU.")
    DEVICE = 'cpu'

## 2. Dataset Acquisition & Extraction
Upload `MachineLearningCSV.zip` to Colab or download it directly.

In [ ]:
import zipfile

ZIP_PATH = "MachineLearningCSV.zip"
EXTRACT_DIR = "data/cic_ids2017/raw"
os.makedirs(EXTRACT_DIR, exist_ok=True)

if os.path.exists(ZIP_PATH):
    print(f"Extracting {ZIP_PATH}...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)
    print("Extraction complete!")
else:
    print(f"Note: Place {ZIP_PATH} in current directory. If already extracted, verifying CSVs...")

csv_files = sorted(glob.glob(os.path.join(EXTRACT_DIR, "**", "*.csv"), recursive=True))
print(f"Found {len(csv_files)} CSV files:")
for f in csv_files:
    print(f"  - {os.path.basename(f)} ({os.path.getsize(f) / (1024*1024):.2f} MB)")

## 3. Forensic Leakage Elimination & Label Normalization

In [ ]:
# Exact leakage columns and true constant features identified in forensic audit
COLUMNS_TO_DROP = [
    "Destination Port",      # Testbed target port memorization
    "Fwd Header Length.1",   # Redundant duplicate
    "Bwd PSH Flags",         # 8 Constant columns (min == max == 0 across all 2.83M rows)
    "Bwd URG Flags",
    "Fwd Avg Bytes/Bulk",
    "Fwd Avg Packets/Bulk",
    "Fwd Avg Bulk Rate",
    "Bwd Avg Bytes/Bulk",
    "Bwd Avg Packets/Bulk",
    "Bwd Avg Bulk Rate"
]

from sklearn.model_selection import train_test_split

print("Starting streaming ingestion and cleaning...")
malicious_chunks = []
benign_chunks = []

for idx, fpath in enumerate(csv_files, 1):
    fname = os.path.basename(fpath)
    print(f"[{idx}/{len(csv_files)}] Processing {fname}...")
    for chunk in pd.read_csv(fpath, chunksize=100000, low_memory=False):
        chunk.columns = [c.strip() for c in chunk.columns]
        lbl_col = "Label" if "Label" in chunk.columns else chunk.columns[-1]
        chunk["original_label"] = chunk[lbl_col].astype(str).str.strip()
        chunk["is_malicious"] = (chunk["original_label"].str.upper() != "BENIGN").astype(np.int32)
        
        mal = chunk[chunk["is_malicious"] == 1]
        ben = chunk[chunk["is_malicious"] == 0]
        
        if len(mal) > 0:
            mal_sampled = mal.sample(n=min(len(mal), 35000), random_state=42) if len(mal) > 35000 else mal
            malicious_chunks.append(mal_sampled)
        if len(ben) > 0:
            ben_sampled = ben.sample(n=min(len(ben), 35000), random_state=42)
            benign_chunks.append(ben_sampled)

df_mal = pd.concat(malicious_chunks, ignore_index=True)
df_ben = pd.concat(benign_chunks, ignore_index=True)
df_all = pd.concat([df_mal, df_ben], ignore_index=True).sample(frac=1.0, random_state=42).reset_index(drop=True)

print(f"Total working flows: {len(df_all):,} (Malicious: {len(df_mal):,}, Benign: {len(df_ben):,})")

# Drop leakage/constant columns
drops = [c for c in COLUMNS_TO_DROP if c in df_all.columns]
df_clean = df_all.drop(columns=drops + ["Label"], errors="ignore")

feature_cols = [c for c in df_clean.columns if c not in ["is_malicious", "original_label"]]
print(f"Selected behavioral feature count: {len(feature_cols)}")

# Numeric cleaning: replace inf, impute median, downcast float32
imputations = {}
for col in feature_cols:
    s = pd.to_numeric(df_clean[col], errors='coerce')
    s = s.replace([np.inf, -np.inf], np.nan)
    med = float(s.median()) if not np.isnan(s.median()) else 0.0
    imputations[col] = med
    df_clean[col] = s.fillna(med).astype(np.float32)

# Split: Train (70%), Val (15%), Test (15%)
train_df, temp_df = train_test_split(df_clean, test_size=0.30, random_state=42, stratify=df_clean["is_malicious"])
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42, stratify=temp_df["is_malicious"])

print(f"Train set: {len(train_df):,} | Val set: {len(val_df):,} | Hold-out Test set: {len(test_df):,}")

## 4. Hardware-Accelerated P6 Training (XGBoost on T4 GPU)

In [ ]:
import pickle

X_train = train_df[feature_cols].values.astype(np.float32)
y_train = train_df["is_malicious"].values.astype(np.int32)

X_val = val_df[feature_cols].values.astype(np.float32)
y_val = val_df["is_malicious"].values.astype(np.int32)

X_test = test_df[feature_cols].values.astype(np.float32)
y_test = test_df["is_malicious"].values.astype(np.int32)

params = {
    'n_estimators': 250,
    'max_depth': 6,
    'learning_rate': 0.08,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 3,
    'gamma': 0.1,
    'tree_method': 'hist',
    'device': DEVICE,
    'eval_metric': 'logloss',
    'random_state': 42
}

print("Training P6 XGBoost Model...")
start_train = time.time()
p6_model = xgb.XGBClassifier(**params)
p6_model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    verbose=50
)
train_duration = round(time.time() - start_train, 2)
print(f"[SUCCESS] Training completed in {train_duration}s on {DEVICE}!")

## 5. Comprehensive Multi-Metric Evaluation

In [ ]:
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score, average_precision_score,
    matthews_corrcoef, brier_score_loss, confusion_matrix
)

p_test = p6_model.predict_proba(X_test)[:, 1]
y_pred = (p_test >= 0.5).astype(int)

metrics = {
    "ROC-AUC": roc_auc_score(y_test, p_test),
    "PR-AUC": average_precision_score(y_test, p_test),
    "Accuracy": accuracy_score(y_test, y_pred),
    "Balanced Accuracy": balanced_accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1": f1_score(y_test, y_pred),
    "MCC": matthews_corrcoef(y_test, y_pred),
    "Brier Score": brier_score_loss(y_test, p_test)
}

print("=" * 50)
print("P6 HOLD-OUT TEST SET METRICS")
print("=" * 50)
for k, v in metrics.items():
    print(f"{k:20s}: {v:.4f}")

cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix (TN, FP / FN, TP):")
print(cm)

## 6. Attack-Category Breakdown & Model Export

In [ ]:
test_df["pred_prob"] = p_test
test_df["pred_label"] = y_pred

breakdown = []
for cat, grp in test_df.groupby("original_label"):
    is_atk = int(grp["is_malicious"].iloc[0] == 1)
    rate = (grp["pred_label"] == is_atk).mean()
    breakdown.append({
        "Category": cat,
        "Class": "Attack" if is_atk else "Benign",
        "Count": len(grp),
        "Accuracy/Recall": f"{rate:.2%}",
        "Mean Prob": f"{grp['pred_prob'].mean():.4f}"
    })

print(pd.DataFrame(breakdown).to_string(index=False))

# Export artifacts
os.makedirs("models/p6", exist_ok=True)
model_out = "models/p6/CyberOptRQ_P6_CIC2017_XGBoost_v1.pkl"
with open(model_out, "wb") as f:
    pickle.dump(p6_model, f)
print(f"\n[SAVED] Exported model artifact to {model_out}")